# Train the CPU weather blend

Reviewed by Tomislav Novosel and the team on 22 October 2026. Prepared with AI assistance. See [AI use](../docs/AI_USE.md).

## Use the selected settings

The CPU blend has MAE 13,494 in the frozen summer check. It has slightly worse MAE and better RMSE than the GRU blend. Use the original settings; do not tune on summer results.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/review_models.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the project.')
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'mushrooms'), str(ROOT / 'citibike'), str(ROOT)]
import pandas as pd
from IPython.display import display
from selection import estimator_for_record
from weather_search import load_weather_features
frozen = json.loads((ROOT / 'reports/research/frozen_finalists.json').read_text())
entry = next(r for r in frozen['finalists'] if r['run_id'] == 'blend_2023_history_and_forecasts')
display(pd.DataFrame([{'family': r['family'], 'settings': r['configuration']} for r in entry['configuration']['members']]))
print('Weights:', entry['configuration']['weights'])

,family,settings
0,weather_svr,"{'feature_set': 'weather_full', 'target': 'dir..."
1,weather_hist,"{'feature_set': 'weather_derived_full', 'targe..."


Weights: [0.75, 0.25]


## Train from earlier dates

SVR uses scaled inputs. Histogram boosting adds corrections. Their fixed weighted average gives 75% to SVR and 25% to boosting. Reproduce on the 153 training targets through June.

In [2]:
from threadpoolctl import threadpool_limits
import joblib
X, y, _ = load_weather_features()
model = estimator_for_record(entry)
with threadpool_limits(limits=2):
    model.fit(X, y)
OUT = ROOT / 'reports/reproduction/citibike'
OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(model, OUT / 'weather_cpu.joblib', compress=3)
print('Reproduction saved. Frozen app artifact remains unchanged.')

Reproduction saved. Frozen app artifact remains unchanged.


## Check inference

This June example belongs to the training side. It checks that inputs become a prediction. It is not an accuracy test.

In [3]:
from src.review_models import bike_example, bike_frame
_, example = bike_frame(bike_example('weather_cpu'))
with threadpool_limits(limits=1):
    print('Example prediction:', round(float(model.predict(example)[0])))
print('Use notebook 15 for the saved development comparison.')

Example prediction: 124882
Use notebook 15 for the saved development comparison.
